In [ ]:
# %% [markdown]
# # UCS420: Cognitive Computing Lab Assignment 10
# **Title:** NLP using Python-II (Feature Extraction, Sentiment Analysis, and Text Generation)

# %%
# Cell 0: Setup and Environment Requirements
import sys
import subprocess

# Function to ensure required packages are installed
required_packages = ['nltk', 'scikit-learn', 'textblob', 'wordcloud', 'matplotlib', 'tensorflow', 'pandas', 'numpy']
for package in required_packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import nltk

# Download required NLTK datasets
for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'vader_lexicon']:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Error downloading {resource}: {e}")

print("Environment setup successfully!")

In [ ]:
# %% [markdown]
# ## Question 1: Basic Text Preprocessing, Tokenization, and Frequency Distribution

# %%
import re
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import pandas as pd

# 1. Unique paragraph (5-6 sentences about Technology / AI)
original_text = (
    "Artificial intelligence is rapidly transforming modern human society in remarkable ways. "
    "Smart algorithms can analyze complex medical images and assist doctors in diagnosing diseases early. "
    "Autonomous vehicles rely heavily on deep neural networks to navigate busy city streets safely. "
    "Natural language processing enables seamless communication between humans and computers. "
    "However, we must address ethical concerns regarding privacy, transparency, and data bias. "
    "The future of technology depends on responsible innovation and continuous human collaboration."
)

print("--- Original Paragraph ---")
print(original_text)
print("\n" + "="*60 + "\n")

# 1. Lowercase & Punctuation removal using re
cleaned_text = original_text.lower()
cleaned_text_no_punct = re.sub(r'[^\w\s]', '', cleaned_text)

print("--- Lowercased & Punctuation Removed ---")
print(cleaned_text_no_punct)
print("\n" + "="*60 + "\n")

# 2. Tokenize into words and sentences
sentences = sent_tokenize(original_text)
words_nltk = word_tokenize(cleaned_text_no_punct)

print(f"Sentence Count: {len(sentences)}")
print("Sentences:", sentences)
print("\n" + "="*60 + "\n")

# 3. Compare Python split() vs NLTK word_tokenize()
# Demonstrating on raw text with punctuation to show real structural differences
raw_sample = "AI isn't just a trend; it's transforming health-care systems."

split_tokens = raw_sample.lower().split()
nltk_tokens = word_tokenize(raw_sample.lower())

print("--- Tokenization Comparison on Raw Sample ---")
print("Raw Sample Text :", raw_sample)
print("Python split()  :", split_tokens)
print("NLTK tokenize() :", nltk_tokens)
print("\nKey Differences:")
print("- Python split(): Splits strictly by whitespace. Keeps punctuation attached to words (e.g., 'trend;', 'systems.').")
print("- NLTK word_tokenize(): Uses treebank rules. Separates punctuation into distinct tokens and handles contractions (e.g., 'isn't' -> 'is', \"n't\").")
print("\n" + "="*60 + "\n")

# 4. Remove Stopwords
stop_words = set(stopwords.words('english'))
filtered_words = [w for w in words_nltk if w not in stop_words]

print("--- Words After Stopword Removal ---")
print(filtered_words)
print("\n" + "="*60 + "\n")

# 5. Word Frequency Distribution (excluding stopwords)
fdist = FreqDist(filtered_words)
print("--- Word Frequency Distribution (Top 10) ---")
for word, freq in fdist.most_common(10):
    print(f"{word:<20}: {freq}")

In [ ]:
# %% [markdown]
# ## Question 2: Regex Filtering, Stemming, and Lemmatization Analysis

# %%
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import pandas as pd

# Using paragraph from Q1
# 1. Extract all words with ONLY alphabets using re.findall()
alphabetic_words = re.findall(r'\b[a-zA-Z]+\b', original_text.lower())

# 2. Remove stopwords
stop_words = set(stopwords.words('english'))
filtered_alphabetic = [word for word in alphabetic_words if word not in stop_words]

# 3. Perform Stemming with PorterStemmer
stemmer = PorterStemmer()
stemmed_words = [stemmer.stem(word) for word in filtered_alphabetic]

# 4. Perform Lemmatization with WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_alphabetic]

# Display Side-by-Side Comparison Table
comparison_df = pd.DataFrame({
    'Original Word': filtered_alphabetic,
    'Stemmed (Porter)': stemmed_words,
    'Lemmatized (WordNet)': lemmatized_words
})

print("--- Stemming vs Lemmatization Comparison Table (Sample First 15 Words) ---")
print(comparison_df.head(15).to_string(index=False))

# 5. Explanation Output
print("\n" + "="*70)
print("--- STEMMING VS LEMMATIZATION COMPARISON & ANALYSIS ---")
print("="*70)
explanation = """
1. Stemming (PorterStemmer):
   - Method: Rule-based heuristic slicing of word suffixes (e.g., '-ing', '-ly', '-es').
   - Output: May produce non-dictionary base stems (e.g., 'society' -> 'societi', 'early' -> 'earli').
   - Pros: Extremely fast, computationally lightweight.
   - Cons: Can cause over-stemming or under-stemming, losing exact semantic meaning.

2. Lemmatization (WordNetLemmatizer):
   - Method: Uses a vocabulary/dictionary (WordNet) and morphological analysis.
   - Output: Always produces valid dictionary root forms ('lemmas') (e.g., 'society' -> 'society').
   - Pros: Preserves context, semantic precision, and readable grammatical words.
   - Cons: Slower, requires vocabulary lookup tables and POS tagging for best results.

WHEN TO PREFER WHICH:
- Prefer Stemming for: Real-time search engines, large-scale indexing, fast text classification, or resource-constrained environments where processing speed matters more than dictionary accuracy.
- Prefer Lemmatization for: Chatbots, Sentiment Analysis, Question-Answering, Summarization, and translation tasks where semantic accuracy and exact meaning are critical.
"""
print(explanation)

In [ ]:
# %% [markdown]
# ## Question 3: Feature Extraction using Bag of Words (BoW) and TF-IDF

# %%
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 3 Short texts (Product / Tech Reviews)
texts = [
    "The camera quality on this new smartphone is exceptionally brilliant and sharp.",
    "Battery life on this laptop is weak and disappointing for daily work.",
    "This smartphone offers great performance with brilliant battery efficiency."
]

print("--- Input Texts ---")
for i, text in enumerate(texts, 1):
    print(f"Text {i}: {text}")
print("\n" + "="*60 + "\n")

# 1. CountVectorizer (Bag of Words)
count_vectorizer = CountVectorizer(stop_words='english')
bow_matrix = count_vectorizer.fit_transform(texts)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=count_vectorizer.get_feature_names_out(), index=['Text 1', 'Text 2', 'Text 3'])

print("--- Bag of Words (BoW) Representation ---")
print(bow_df)
print("\n" + "="*60 + "\n")

# 2. TfidfVectorizer (TF-IDF Scores)
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out(), index=['Text 1', 'Text 2', 'Text 3'])

print("--- TF-IDF Matrix ---")
print(tfidf_df.round(3))
print("\n" + "="*60 + "\n")

# 3. Top 3 Keywords per text using TF-IDF
print("--- Top 3 Keywords per Text (TF-IDF Interpretation) ---")
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

for i, text in enumerate(texts):
    row = tfidf_matrix.toarray()[i]
    top_indices = row.argsort()[-3:][::-1]
    top_words = feature_names[top_indices]
    top_scores = row[top_indices]
    
    keywords_str = ", ".join([f"'{w}' ({s:.3f})" for w, s in zip(top_words, top_scores)])
    print(f"\nText {i+1}: \"{text}\"")
    print(f"  Top 3 Keywords: {keywords_str}")
    print(f"  Interpretation: The word(s) '{top_words[0]}' hold the highest uniqueness/importance score specific to Text {i+1}.")

In [ ]:
# %% [markdown]
# ## Question 4: Text Similarity Analysis - Jaccard vs Cosine Similarity

# %%
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Two short texts describing AI vs Blockchain
text_a = """
Artificial Intelligence uses machine learning algorithms to analyze large datasets.
It enables computers to learn patterns and make autonomous predictions accurately.
Neural networks simulate human cognitive functions to solve complex problems.
AI systems automate repetitive tasks and improve efficiency across industries.
"""

text_b = """
Blockchain technology provides a decentralized distributed ledger for transactions.
It records data across peer to peer networks with cryptographic security.
Smart contracts execute automatically when predefined conditions are satisfied.
Blockchain ensures immutability, transparency, and trust without central authorities.
"""

# Preprocessing helper function
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    return tokens

tokens_a = preprocess(text_a)
tokens_b = preprocess(text_b)

# 2a. Jaccard Similarity using Sets
set_a = set(tokens_a)
set_b = set(tokens_b)

intersection = set_a.intersection(set_b)
union = set_a.union(set_b)
jaccard_sim = len(intersection) / len(union)

# 2b. Cosine Similarity using TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform([text_a, text_b])
cosine_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

print("--- SIMILARITY METRICS RESULTS ---")
print(f"Common Words in Intersection : {intersection}")
print(f"Jaccard Similarity Score    : {jaccard_sim:.4f}")
print(f"Cosine Similarity Score     : {cosine_sim:.4f}")
print("\n" + "="*70)

# 2c. Analysis & Insights
analysis = """
--- COMPARATIVE ANALYSIS: JACCARD VS COSINE SIMILARITY ---

1. Jaccard Similarity (Score: {:.4f}):
   - Evaluates set membership (unweighted exact word overlap divided by total unique union size).
   - Treats all overlapping words equally regardless of their domain importance (e.g., 'and', 'to', 'for').
   - Ignores term frequency completely.

2. Cosine Similarity with TF-IDF (Score: {:.4f}):
   - Measures vector angles in high-dimensional TF-IDF feature space.
   - Assigns lower weight to common filler words and higher weight to distinct domain terms.
   - Accounting for term importance and word frequency makes it far more robust.

WHICH METRIC GIVES BETTER INSIGHTS?
Cosine Similarity provides significantly better insights. AI and Blockchain discuss completely different technical domains. 
Cosine Similarity correctly captures this high semantic divergence, giving a low score based on domain terms, 
whereas Jaccard is sensitive to document lengths and coincidental non-informative word overlap.
""".format(jaccard_sim, cosine_sim)

print(analysis)

In [ ]:
# %% [markdown]
# ## Question 5: Sentiment Analysis (TextBlob/VADER) and Positive Word Cloud Generation

# %%
import pandas as pd
from textblob import TextBlob
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Dataset of short product/service reviews
reviews = [
    "This laptop is amazingly fast, sleek, and has an incredible battery life! Absolutely love it.",
    "The customer service was terrible, slow, and completely unhelpful. Very disappointing experience.",
    "The package arrived on time. The product works as expected, nothing special.",
    "Exceptional build quality and fantastic sound performance! Highly recommended to everyone.",
    "Extremely bad experience. The phone overheats within five minutes and crashes constantly.",
    "Decent product for the price. Performs average functions without major issues."
]

vader_analyzer = SentimentIntensityAnalyzer()
results = []

for review in reviews:
    # TextBlob
    tb = TextBlob(review)
    polarity = tb.sentiment.polarity
    subjectivity = tb.sentiment.subjectivity
    
    # Classification based on Polarity threshold
    if polarity > 0.1:
        sentiment_label = "Positive"
    elif polarity < -0.1:
        sentiment_label = "Negative"
    else:
        sentiment_label = "Neutral"
        
    # VADER Compound Score
    vader_score = vader_analyzer.polarity_scores(review)['compound']
    
    results.append({
        'Review': review,
        'Polarity (TextBlob)': round(polarity, 3),
        'Subjectivity (TextBlob)': round(subjectivity, 3),
        'VADER Compound': round(vader_score, 3),
        'Sentiment': sentiment_label
    })

df_reviews = pd.DataFrame(results)

print("--- SENTIMENT ANALYSIS RESULTS ---")
print(df_reviews[['Review', 'Polarity (TextBlob)', 'Subjectivity (TextBlob)', 'Sentiment']].to_string(index=False))
print("\n" + "="*60 + "\n")

# Combine all POSITIVE reviews for Word Cloud
positive_reviews = df_reviews[df_reviews['Sentiment'] == 'Positive']['Review']
combined_positive_text = " ".join(positive_reviews)

print("Combined Text for Positive WordCloud:")
print(combined_positive_text)

# Generate and Display WordCloud
wordcloud = WordCloud(
    width=800, 
    height=400, 
    background_color='white',
    colormap='Greens',
    max_words=100
).generate(combined_positive_text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Positive Customer Reviews', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# %% [markdown]
# ## Question 6: Text Generation using Keras Tokenizer and LSTM / Neural Network

# %%
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Training corpus (~100 words on AI & Deep Learning)
training_corpus = """
Natural language processing and deep learning are revolutionizing modern computer science.
Deep neural networks allow computers to analyze textual data and uncover hidden patterns.
Machine learning models learn continuous representations from sequence data effectively.
Language models predict the next sequence of words based on contextual probabilities.
Artificial intelligence empowers intelligent decision making across modern automated systems.
Recurrent networks and transformers drive cutting edge advancements in text processing.
Future technology relies heavily on intelligent algorithms and powerful data architectures.
"""

# 1. Tokenization using Keras Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_corpus])
total_words = len(tokenizer.word_index) + 1

print(f"Total Unique Words (Vocabulary Size): {total_words}")

# Create input sequences for N-gram training
input_sequences = []
for line in training_corpus.strip().split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Create predictors and label
X, y = input_sequences[:, :-1], input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# 2. Build LSTM Model
model = Sequential([
    Embedding(input_dim=total_words, output_dim=50, input_length=max_sequence_len - 1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print("\n--- Model Summary ---")
model.summary()

# Train Model
print("\nTraining the model...")
model.fit(X, y, epochs=150, verbose=0)
print("Model training complete!\n")

# 3. Helper Function to Generate Next N Words
def generate_text(seed_text, next_words, model, max_sequence_len):
    output_text = seed_text
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted_probs = model.predict(token_list, verbose=0)[0]
        predicted_idx = np.argmax(predicted_probs)
        
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_idx:
                output_word = word
                break
        output_text += " " + output_word
    return output_text

# Generate 2-3 lines starting from different seed words
seed_words = ["natural language", "deep neural", "artificial intelligence"]

print("="*60)
print("--- GENERATED TEXT OUTPUTS ---")
print("="*60)

for seed in seed_words:
    generated_line = generate_text(seed, next_words=7, model=model, max_sequence_len=max_sequence_len)
    print(f"Seed: '{seed}' --> Generated Output:\n  \"{generated_line}\"\n")